# Staged Charts: Limits per Stage

A process that changes on purpose has more than one level. The coffee shop in this tutorial changed
three times in sixteen weeks: a new espresso machine in week 8, a new point-of-sale (POS) system in
week 12, and new baristas from week 14. One set of limits across all sixteen weeks answers *did
something change?* Limits per stage answer the next question: *is each stage stable on its own, and at
what level?* That is what `phased=True` does.

**What You'll Learn**

- Derive a stage column from a time column with `.bin()`
- Compare one set of limits with limits per stage on the Xbar and S charts
- Read the per-stage numbers with `phase_statistics()`
- Put every reading on a phased X/mR chart
- Pool a second factor into the subgroup, or chart each of its levels on its own

## Setup

The coffee-shop data ships with the library: drink wait time in seconds, four readings a day, Monday to
Saturday, for 16 weeks. One `.bin()` call turns the `week` column into a stage. The cut points 8, 12
and 14 give four stages: weeks 1-7, 8-11, 12-13 and 14-16 (each interval includes its lower edge).

In [1]:
from processbehavior import load_coffee_shop

pb = load_coffee_shop().bin(
    'week', method='breaks', breaks=[8, 12, 14],
    bin_labels=['baseline', 'new machine', 'new POS', 'new baristas'],
    label='stage',
)
pb.data.head()

,date,year_week,week,day_of_week,hour,daypart,wait_sec
0,2026-01-05,2026-W01,1,Mon,8,Peak,239
1,2026-01-05,2026-W01,1,Mon,10,Off-peak,225
2,2026-01-05,2026-W01,1,Mon,13,Peak,234
3,2026-01-05,2026-W01,1,Mon,16,Off-peak,218
4,2026-01-06,2026-W01,1,Tue,8,Peak,241


`stage` isn't in `pb.data` yet: `.bin()` attaches a derivation, and `formulate()` evaluates it. The
result is an ordered categorical, so the stages keep the order of `bin_labels` instead of sorting
alphabetically.

## Formulate

Stage is the factor and the day is the time axis, so each day's four readings form one subgroup.

In [2]:
study = pb.formulate(response='wait_sec', factors=['stage'], time='date')
print(study.design())

Design Report (1 factors)
  Design-state lineage:
    PDS (Planned):    no plan supplied
    SDS (Sampling):   4 (Incomplete, No Singletons) — 288 empty cells
    ADS (Analytical): 1 (Full Replication)
  Min cell size: 4 | K: 4 | T: 96 | R: 96 | N: (min=4, median=4.0, max=4)

  Factors:
    stage: observed=['baseline', 'new POS', 'new baristas', 'new machine']

  Structure: Complete structure
  Series length: T=96. Sigma rests on within-cell replication.

  Available analyses (ADS 1):
    Primary: Histogram, Xbar *, S, X, mR
    R2: S, X
    R3: Xbar, S
    R4: Xbar, S
    R5: Xbar, S
    R6: Xbar, S
    Methods: Capability, Loss Function, Maximum Information


The sampling line reports an incomplete design with 288 empty cells. For a staged study that is
expected, not a data problem: every day belongs to exactly one stage, so most of the stage × date grid
is empty by construction. The analytical design state is 1 (full replication: four readings in every
subgroup), and that is what the charts use.

## One set of limits

First, the charts that treat all sixteen weeks as one process.

In [3]:
whole = study.execute(chart='Xbar', companion=True)
whole.plot(chart='Xbar', title='Wait time: one set of limits for 16 weeks').show()
whole.plot(chart='S', title='Spread: one set of limits for 16 weeks')

The center line, 220.8 seconds, sits between the baseline level and the level after the new machine,
so it describes neither. Three days fall outside the Xbar limits, and two days late in the series fall
above the S limit: the spread grew. These charts find the changes, but they can't say what each stage
looks like on its own.

## Limits per stage

`phased=True` gives each stage its own center line and limits on the same chart, drawn as steps with a
divider at each stage change.

In [4]:
staged = study.execute(chart='Xbar', phased=True, companion=True)
staged.plot(chart='Xbar', title='Wait time: limits per stage').show()
staged.plot(chart='S', title='Spread: limits per stage')

Each stage is computed as a separate analysis. Its center is the unweighted mean of its subgroup
means, its S̄ is the mean of its subgroup standard deviations, and its σ̂ and limits follow from those
(10-12 manual Eqs 11-5, 11-8, 11-9, 11-16, 11-17). Nothing is pooled across stages, so each stage is
judged against its own data.

## The numbers behind the steps

`phase_statistics()` returns one row per stage: the number of subgroups, the subgroup size `N`, the
center, S̄, σ̂ and the limits.

In [5]:
staged.phase_statistics('Xbar')

,phase,subgroups,N,center,s_bar,sigma_hat,lpl,upl
0,baseline,42,4,241.256,24.626,26.729,201.162,281.350
1,new machine,24,4,200.615,21.280,23.098,165.968,235.261
2,new POS,12,4,220.500,26.196,28.433,177.850,263.150
3,new baristas,18,4,200.028,32.110,34.852,147.750,252.306


In [6]:
staged.phase_statistics('S')

,phase,subgroups,N,center,s_bar,sigma_hat,lpl,upl
0,baseline,42,4,24.626,24.626,26.729,0.0,55.804
1,new machine,24,4,21.280,21.280,23.098,0.0,48.222
2,new POS,12,4,26.196,26.196,28.433,0.0,59.361
3,new baristas,18,4,32.110,32.110,34.852,0.0,72.762


Read down the stages:

- **New machine:** the center fell from 241.3 to 200.6 seconds and stayed there.
- **New POS:** a learning curve. The center rose to 220.5 while the staff learned the system.
- **New baristas:** the center is back at 200.0, but S̄ rose to 32.1 from 21.3 after the machine. The
  average wait held while the waits became less consistent. Averages hide variation.

## Signals

On a phased chart, Rule 1 runs against each stage's own limits. The run rules are off, because their
zones change from stage to stage.

In [7]:
staged.detect_signals()

{'Xbar': SignalResult(violations=1, flagged_obs=1, chart='Xbar'),
 'S': SignalResult(violations=0, flagged_obs=0, chart='S')}

In [8]:
staged.chart_table('Xbar').query("signal != ''")

,subgroup,phase,n,value,center,lpl,upl,signal
67,new POS_2026-03-23,new POS,4,280.0,220.5,177.85,263.15,↑


One day stands out within its own stage: 23 March, the first day of the new POS, with an average wait
of 280 seconds against that stage's upper limit of 263.2. The two barista days that broke the single S
limit now sit inside their stage's limits, because the wider spread is that stage's normal. The two
views answer different questions: use one set of limits to find the changes, and limits per stage to
judge each stage.

## Every reading: phased X/mR

The same idea works on individual readings. `chart='x'` with `by=[]` puts all 384 readings on one
chart, and `phased=True` gives each stage its own limits. The moving range restarts at each stage, so
no range spans a stage change.

In [9]:
readings = study.execute(chart='x', by=[], phased=True, companion=True)
readings.plot(chart='X', title='Every reading: limits per stage').show()
readings.plot(chart='mR', title='Moving range: limits per stage')

In [10]:
readings.phase_statistics('X')

,phase,points,center,mR_bar,lpl,upl
0,baseline,168,241.256,37.856,140.574,341.938
1,new machine,96,200.615,30.032,120.743,280.486
2,new POS,48,220.500,33.809,130.584,310.416
3,new baristas,72,200.028,41.887,88.625,311.430


The moving range tells the barista story too: the stage's mR̄ is 41.9 seconds against 30.0 after the
new machine, and two ranges exceed even that stage's own upper limit.

## A second factor: pool it or split by it

The rush hours (08:00 and 13:00, the `Peak` daypart) are slower than the quiet ones. With `daypart` as a
second factor, there are two useful views.

To pool the dayparts into each day's subgroup, name the stage and the time in `by=`. The phases are on
the stage alone, and the numbers equal the one-factor study above:

In [11]:
two = pb.formulate(response='wait_sec', factors=['stage', 'daypart'], time='date')
two.execute(chart='Xbar', by=['stage', 'date'], phased=True).phase_statistics('Xbar')

,phase,subgroups,N,center,s_bar,sigma_hat,lpl,upl
0,baseline,42,4,241.256,24.626,26.729,201.162,281.350
1,new machine,24,4,200.615,21.280,23.098,165.968,235.261
2,new POS,12,4,220.500,26.196,28.433,177.850,263.150
3,new baristas,18,4,200.028,32.110,34.852,147.750,252.306


To chart each daypart on its own, put `daypart` in `by=`: one X chart per daypart, each phased on the
stage.

In [12]:
two.execute(chart='x', by=['daypart'], phased=True).plot(facet=True, ncols=1, title='Each daypart: limits per stage')

## Saving a chart

Every chart above is one chain from the study to the figure. End the chain with `.save_image()` to
write a PNG, which needs the `images` extra (`pip install 'processbehavior[images]'`), or with
`.save_html()`, which needs nothing extra:

```python
(study.execute(chart='Xbar', phased=True, companion=True)
      .plot(chart='Xbar', title='Wait time: limits per stage')
      .save_image(filepath='wait-time-by-stage.png'))
```

## Things to Know

- **Stage order follows the factor's level order.** An ordered categorical, which is what `.bin()`
  returns, keeps its order. Plain text labels sort alphabetically, so label them to sort (`1 before`,
  `2 after`).
- **A stage is a level, not a stretch of time.** If a level comes back later (A, then B, then A again),
  both runs of A become one phase placed before B. Give a returning condition its own label
  (`3 A again`).
- **Phased Xbar/S refuses** `by=[time]` (that view is already one chart per stage), residual values and
  calibrations.
- **A stage with no replicated subgroup** keeps its points but has no limits, and is named in
  `metadata['insufficient_phases']`.

## Next Steps

- [Xbar-S Analysis](xbar-s-analysis.ipynb): the S chart in its own right
- [Stratified Analysis](stratified-analysis.ipynb): one chart per stream, and `focus()`
- [Derived Variables](derived-variables.ipynb): more ways to derive a factor before formulating
- [Chart Types: Staged Xbar/S](../user-guide/chart-types.md#staged-xbar-s): the reference for phased charts